In [1]:
import os
import numpy as np
import pandas as pd

# Simulate one raw GBM-driven OHLCV stock
We are simulating one stock path for the `n_steps` trading days. In particular>
* `Close`: is generated from a GBM-style process
* `Open`: comes from an overnight GBM increment
* `High`: is a positive excursion around `Open`
* `Low`: is a positive excursion around `Close`
* `Volume`: is generated separately, with larger values on larger-move days

In [2]:
def simulate_gbm_ohlcv_path(
    n_steps=10081,              # raw rows; after preprocessing -> 10080
    s0=100.0,
    mu=0.08,
    sigma=0.25,
    dt=1/252,
    overnight_scale=0.25,
    intraday_high_scale=0.6,
    intraday_low_scale=0.6,
    vol0=1_000_000.0,
    vol_sigma=0.20,
    vol_ret_coupling=6.0,
    start_date="1986-03-13",
    seed=None,
):
    rng = np.random.default_rng(seed)

    dates = pd.bdate_range(start=start_date, periods=n_steps)

    open_prices = np.zeros(n_steps, dtype=np.float64)
    high_prices = np.zeros(n_steps, dtype=np.float64)
    low_prices = np.zeros(n_steps, dtype=np.float64)
    close_prices = np.zeros(n_steps, dtype=np.float64)
    volumes = np.zeros(n_steps, dtype=np.float64)

    prev_close = s0
    log_vol_prev = np.log(vol0)

    for t in range(n_steps):
        # Overnight move: previous close -> today's open
        z_overnight = rng.normal()
        dt_overnight = dt * overnight_scale
        r_overnight = (
            (mu - 0.5 * sigma**2) * dt_overnight
            + sigma * np.sqrt(dt_overnight) * z_overnight
        )
        open_t = prev_close * np.exp(r_overnight)

        # Intraday move: open -> close
        z_intraday = rng.normal()
        dt_intraday = dt * (1.0 - overnight_scale)
        r_intraday = (
            (mu - 0.5 * sigma**2) * dt_intraday
            + sigma * np.sqrt(dt_intraday) * z_intraday
        )
        close_t = open_t * np.exp(r_intraday)

        # Intraday high/low excursions
        daily_scale = sigma * np.sqrt(dt)
        up_exc = abs(rng.normal(0.0, intraday_high_scale * daily_scale))
        down_exc = abs(rng.normal(0.0, intraday_low_scale * daily_scale))

        high_base = max(open_t, close_t)
        low_base = min(open_t, close_t)

        high_t = high_base * np.exp(up_exc)
        low_t = low_base * np.exp(-down_exc)

        # Safety
        high_t = max(high_t, open_t, close_t)
        low_t = min(low_t, open_t, close_t)

        # Volume process
        cc_ret = np.log(close_t / prev_close)
        vol_shock = rng.normal(scale=vol_sigma)
        log_vol_t = (
            0.98 * log_vol_prev
            + 0.02 * np.log(vol0)
            + vol_ret_coupling * abs(cc_ret)
            + vol_shock
        )
        volume_t = max(np.exp(log_vol_t), 1.0)

        open_prices[t] = open_t
        high_prices[t] = high_t
        low_prices[t] = low_t
        close_prices[t] = close_t
        volumes[t] = volume_t

        prev_close = close_t
        log_vol_prev = log_vol_t

    df = pd.DataFrame({
        "Date": dates,
        "Open": open_prices,
        "High": high_prices,
        "Low": low_prices,
        "Close": close_prices,
        "Volume": volumes,
    })

    return df

# Preprocessing
This preprocess is done in order to match the one that is applied in `SP500_downloader_gbm.ipynb`:
* OHCL columns become log-ratios relative to the previous close price
* Volume becomes a log-difference
* each channel is standardized

In [3]:
def preprocess_like_notebook(df_raw):
    data = df_raw.copy().sort_values("Date").reset_index(drop=True)

    ohlc_cols = ["Open", "High", "Low", "Close"]
    prev_close = data["Close"].shift(1)

    # OHLC relative to previous close
    ohlc_log_rets = np.log(data[ohlc_cols].div(prev_close, axis=0))

    # Volume log-difference
    volume_log_rets = np.log(data["Volume"] + 1.0) - np.log(data["Volume"].shift(1) + 1.0)

    processed = pd.concat(
        [ohlc_log_rets, volume_log_rets.rename("Volume")],
        axis=1
    ).dropna()

    arr = processed.to_numpy(dtype=np.float32)
    mean = arr.mean(axis=0)
    std = arr.std(axis=0)

    arr_std = (arr - mean) / (std + 1e-8)

    df_processed = pd.DataFrame(
        arr_std,
        columns=["Open", "High", "Low", "Close", "Volume"]
    )

    # keep Date as first column
    df_processed.insert(0, "Date", data.loc[processed.index, "Date"].dt.strftime("%d/%m/%Y").values)

    return df_processed, {"mean": mean, "std": std}

# Generating fake stocks

## Generating one fake stock
We generate one fake stock with slightly different parameters from the others, which are sampled uniformly in a range. So that each stock will have different, yet constraint:
* initial price
* drift
* volatility
* average volume

In [4]:
def generate_one_fake_stock(
    stock_id,
    n_processed_rows=10080,
    start_date="1986-03-13",
    base_seed=1234,
):
    rng = np.random.default_rng(base_seed + stock_id)

    # Slight parameter variation across fake stocks
    s0 = rng.uniform(20.0, 300.0)
    mu = rng.uniform(0.02, 0.15)
    sigma = rng.uniform(0.10, 0.45)
    vol0 = rng.uniform(1e5, 5e6)

    # +1 because preprocessing drops first row
    n_raw_steps = n_processed_rows + 1

    df_raw = simulate_gbm_ohlcv_path(
        n_steps=n_raw_steps,
        s0=s0,
        mu=mu,
        sigma=sigma,
        dt=1/252,
        vol0=vol0,
        start_date=start_date,
        seed=base_seed + stock_id,
    )

    df_processed, stats = preprocess_like_notebook(df_raw)

    return df_processed, stats

## Save many fake stocks
We save these many fake stocks into individual CSV as to match the previously build model

In [5]:
def save_fake_stock_universe(
    output_dir="../data/fake_individual_gbm/",
    n_stocks=200,
    n_processed_rows=10080,
    start_date="1986-01-01",
    base_seed=1234,
):
    os.makedirs(output_dir, exist_ok=True)

    stats_dict = {}

    for i in range(n_stocks):
        df_processed, stats = generate_one_fake_stock(
            stock_id=i,
            n_processed_rows=n_processed_rows,
            start_date=start_date,
            base_seed=base_seed,
        )

        ticker = f"FAKE_{i+1:04d}"

        first_day = pd.to_datetime(df_processed["Date"], format="%d/%m/%Y").iloc[0].strftime("%Y-%m-%d")
        last_day = pd.to_datetime(df_processed["Date"], format="%d/%m/%Y").iloc[-1].strftime("%Y-%m-%d")

        filename = f"{ticker}_{first_day}_{last_day}_processed.csv"
        filepath = os.path.join(output_dir, filename)

        df_processed.to_csv(filepath, index=False)

        stats_dict[ticker] = stats

    return stats_dict

# Executing it

In [ ]:
stats_dict = save_fake_stock_universe(
    output_dir="../data/fake_individual_gbm/",
    n_stocks=200,
    n_processed_rows=10080,   # 40 * 252
    start_date="1986-01-01",
    base_seed=42,
)

# Saving Stats dictionary
Useful to revert the normalization process

In [ ]:
import pickle
from datetime import datetime
import os

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

file_path = f"../data/fake_individual_gbm/fake_stats_{timestamp}.pkl"

with open(file_path, "wb") as f:
    pickle.dump(stats_dict, f)

NameError: name 'stats_dict' is not defined